We turned token IDs into 768-dimensional vectors: `wte[token_id] + wpe[position]`.  
That gave us a `(seq_len, 768)` matrix — the input to the first transformer block


Now: **how does the model make tokens talk to each other?**

This is the most compute-intensive operation per token during inference. Understanding it deeply is essential for understanding every optimization that follows.

### Setup 
- Load the weights (safetensors) in `weights`.
- Load the tokenizer (breaks input text into chars -> merge rules to find dominant tokens -> map to vocabulary) as `encode(text)` and `decode(token_ids)`.
- Load the embedder (for each token_id -> wte(token_id) + wpe(seq_pos) = embed(seq_pos * d_model) as `embed(token_ids)`.

In [1]:
import json
import struct
import os
import re
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

MODEL_DIR = "gpt2_weights"

# --- Load weights ---
def parse_safetensors(path):
    with open(path, "rb") as f:
        header_len = struct.unpack("<Q", f.read(8))[0]
        header = json.loads(f.read(header_len))
        data_start = 8 + header_len
        tensors = {}
        for name, info in header.items():
            if name == "__metadata__":
                continue
            dtype = {"F32": np.float32, "F16": np.float16}[info["dtype"]]
            start, end = info["data_offsets"]
            f.seek(data_start + start)
            tensors[name] = np.frombuffer(f.read(end - start), dtype=dtype).reshape(info["shape"])
        return tensors

weights = parse_safetensors(os.path.join(MODEL_DIR, "model.safetensors"))

# --- Load tokenizer ---
with open(os.path.join(MODEL_DIR, "vocab.json")) as f:
    vocab = json.load(f)
with open(os.path.join(MODEL_DIR, "merges.txt")) as f:
    merges_raw = f.read().strip().split("\n")[1:]

id_to_token = {v: k for k, v in vocab.items()}
bpe_ranks = {tuple(m.split()): i for i, m in enumerate(merges_raw)}

def bytes_to_unicode():
    bs = list(range(33, 127)) + list(range(161, 173)) + list(range(174, 256))
    cs = bs[:]
    n = 0
    for b in range(256):
        if b not in bs:
            bs.append(b)
            cs.append(256 + n)
            n += 1
    return dict(zip(bs, [chr(c) for c in cs]))

byte_encoder = bytes_to_unicode()
byte_decoder = {v: k for k, v in byte_encoder.items()}
GPT2_PATTERN = re.compile(r"""'s|'t|'re|'ve|'m|'ll|'d| ?\w+| ?[^\s\w]+|\s+(?!\S)|\s+""")

def get_pairs(tokens):
    return set(zip(tokens[:-1], tokens[1:]))

def bpe_merge(tokens):
    while True:
        pairs = get_pairs(tokens)
        if not pairs:
            break
        best = min(pairs, key=lambda p: bpe_ranks.get(p, float("inf")))
        if best not in bpe_ranks:
            break
        first, second = best
        new = []
        i = 0
        while i < len(tokens):
            if i < len(tokens)-1 and tokens[i] == first and tokens[i+1] == second:
                new.append(first + second)
                i += 2
            else:
                new.append(tokens[i])
                i += 1
        tokens = new
    return tokens

def encode(text):
    ids = []
    for chunk in re.findall(GPT2_PATTERN, text):
        uc = [byte_encoder[b] for b in chunk.encode("utf-8")]
        for t in bpe_merge(uc):
            ids.append(vocab[t])
    return ids

def decode(token_ids):
    text = "".join(id_to_token[i] for i in token_ids)
    return bytes([byte_decoder[c] for c in text]).decode("utf-8", errors="replace")

# --- Embedding function from Day 03 ---
wte = weights["wte.weight"]
wpe = weights["wpe.weight"]

def embed(token_ids):
    seq_len = len(token_ids)
    return wte[token_ids] + wpe[:seq_len]

print(f"Weights loaded: {len(weights)} tensors")
print(f"Tokenizer ready: {len(vocab):,} tokens")
print(f"Embedding tables: wte {wte.shape}, wpe {wpe.shape}")

Weights loaded: 160 tensors
Tokenizer ready: 50,257 tokens
Embedding tables: wte (50257, 768), wpe (1024, 768)


## Looking at a transformer block

GPT-2 small has 12 blocks (layers), named `h.0` through `h.11`.

Each block has 3 sublayers:
1. attention layer (multi head)
2. feed forward layer
3. normalisation layer
   

In [2]:
# Inventory the weights in block 0
block0_weights = {k: v.shape for k, v in weights.items() if k.startswith("h.0.")}

print("Weights in transformer block h.0:")
print("=" * 55)

attn_params = 0
mlp_params = 0
norm_params = 0

for name, shape in sorted(block0_weights.items()):
    n = np.prod(shape)
    component = "ATTN" if "attn" in name else "MLP" if "mlp" in name else "NORM"
    if component == "ATTN":
        attn_params += n
    elif component == "MLP":
        mlp_params += n
    else:
        norm_params += n
    print(f"  {name:<35s} {str(shape):<20s} {n:>10,d}  [{component}]")

total = attn_params + mlp_params + norm_params
print(f"\nParameter breakdown per block:")
print(f"  Attention:     {attn_params:>10,d}  ({100*attn_params/total:.1f}%)")
print(f"  Feed-forward:  {mlp_params:>10,d}  ({100*mlp_params/total:.1f}%)")
print(f"  Normalization: {norm_params:>10,d}  ({100*norm_params/total:.1f}%)")
print(f"  Total:         {total:>10,d}")
print(f"\n  x 12 blocks = {total * 12:,d} parameters in transformer blocks")

Weights in transformer block h.0:
  h.0.attn.bias                       (1, 1, 1024, 1024)    1,048,576  [ATTN]
  h.0.attn.c_attn.bias                (2304,)                   2,304  [ATTN]
  h.0.attn.c_attn.weight              (768, 2304)           1,769,472  [ATTN]
  h.0.attn.c_proj.bias                (768,)                      768  [ATTN]
  h.0.attn.c_proj.weight              (768, 768)              589,824  [ATTN]
  h.0.ln_1.bias                       (768,)                      768  [NORM]
  h.0.ln_1.weight                     (768,)                      768  [NORM]
  h.0.ln_2.bias                       (768,)                      768  [NORM]
  h.0.ln_2.weight                     (768,)                      768  [NORM]
  h.0.mlp.c_fc.bias                   (3072,)                   3,072  [MLP]
  h.0.mlp.c_fc.weight                 (768, 3072)           2,359,296  [MLP]
  h.0.mlp.c_proj.bias                 (768,)                      768  [MLP]
  h.0.mlp.c_proj.weight          

## 1. The Attention layer

Let's look at it with the following lenses:

- the attention equation and how they are derived from the input embeddings (seq_len * 786)
- what each of the attention parameters correspond to:
  ```  
  h.0.attn.bias                       (1, 1, 1024, 1024)    1,048,576  [ATTN]
  h.0.attn.c_attn.bias                (2304,)                   2,304  [ATTN]
  h.0.attn.c_attn.weight              (768, 2304)           1,769,472  [ATTN]
  h.0.attn.c_proj.bias                (768,)                      768  [ATTN]
  h.0.attn.c_proj.weight              (768, 768)              589,824  [ATTN]
  ```
- masking
- multi head attention and why

### The Attention equation

The standard form is **scaled dot-product attention** (*Book Figure 2.8*):

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

Where $d_k$ is the dimension of the key vectors.

Think of it like a search engine:
- Q is the search query
    - **Q (Queries):** The embedded representation of the token being generated or updated.  
  *"What am I looking for?"* 
- K is the index of documents
    - **K (Keys):** Representations of all prior tokens.  
  *"What do I contain?"*
- $QK^T$ computes relevance scores (dot product = similarity)
- $\sqrt{d_k}$ prevents scores from getting too large (keeps softmax well-behaved)
- softmax turns scores into probabilities that sum to 1
- V is the document content — weighted sum by relevance
    - **V (Values):** The actual information to retrieve from each prior token.  
  *"If you pick me, here's what you get."*